# 07 — Climate Impact Attribution and Risk Analysis

## Research Notebook

This notebook builds on preprocessing, climate extremes, bias correction, and multi-model ensemble analysis. It translates climate indicators into hazard changes, threshold exceedance, uncertainty, exposure, vulnerability, and risk-oriented summaries.

## Objectives

1. Compare historical and future climate periods.
2. Quantify absolute and relative changes.
3. Analyze threshold exceedance and extreme-event frequency.
4. Quantify model agreement and uncertainty.
5. Analyze annual and seasonal hazards.
6. Estimate trends.
7. Combine hazard, exposure, and vulnerability where suitable data exist.
8. Produce reproducible tables, figures, and NetCDF outputs.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [ ]:
BASE=Path('.')
OUTPUT_DIR=BASE/'outputs'/'attribution'
FIGURE_DIR=OUTPUT_DIR/'figures'
TABLE_DIR=OUTPUT_DIR/'tables'
for d in [OUTPUT_DIR,FIGURE_DIR,TABLE_DIR]: d.mkdir(parents=True,exist_ok=True)
VARIABLE='tasmax'
SCENARIO='rcp26'
BASELINE_START,BASELINE_END='1981-01-01','2000-12-31'
FUTURE_START,FUTURE_END='2081-01-01','2100-12-31'

## 1. Recommended indicators

Temperature: TXx, TNx, TX90p, WSDI. Precipitation: RX1day, RX5day, R95p, R99p, CDD, CWD. Drought: SPEI-1, SPEI-3, SPEI-6, SPEI-12.

In [ ]:
INDICATORS=['TXx','TNx','TX90p','WSDI','RX1day','RX5day','R95p','R99p','CDD','CWD','SPEI1','SPEI3','SPEI6','SPEI12']
print(', '.join(INDICATORS))

## 2. Period selection

In [ ]:
def select_period(data,start,end): return data.sel(time=slice(start,end))
def period_mean(data,start,end): return select_period(data,start,end).mean('time',skipna=True)
def period_median(data,start,end): return select_period(data,start,end).median('time',skipna=True)

## 3. Absolute change

In [ ]:
def absolute_change(data): return period_mean(data,FUTURE_START,FUTURE_END)-period_mean(data,BASELINE_START,BASELINE_END)

## 4. Relative change

In [ ]:
def relative_change(data):
    b=period_mean(data,BASELINE_START,BASELINE_END); f=period_mean(data,FUTURE_START,FUTURE_END)
    return 100*(f-b)/b.where(np.abs(b)>1e-12)

## 5. Model-level change signal

In [ ]:
def model_change_signal(model_data):
    b=model_data.sel(time=slice(BASELINE_START,BASELINE_END)).mean('time')
    f=model_data.sel(time=slice(FUTURE_START,FUTURE_END)).mean('time')
    return f-b

## 6. Ensemble change statistics

In [ ]:
def ensemble_change_statistics(change):
    return xr.Dataset({'mean':change.mean('model'),'median':change.median('model'),'std':change.std('model'),'min':change.min('model'),'max':change.max('model'),'p10':change.quantile(.10,dim='model'),'p90':change.quantile(.90,dim='model')})

## 7. Model agreement

In [ ]:
def positive_fraction(data): return (data>0).mean('model')
def negative_fraction(data): return (data<0).mean('model')
def agreement_category(a): return xr.where(a>=.80,3,xr.where(a>=.60,2,xr.where(a>=.50,1,0)))

## 8. Threshold exceedance frequency

In [ ]:
def exceedance_frequency(data,threshold,direction='above'):
    if direction=='above': return (data>threshold).mean('time')
    if direction=='below': return (data<threshold).mean('time')
    raise ValueError("direction must be 'above' or 'below'")

## 9. Threshold exceedance count

In [ ]:
def exceedance_count(data,threshold,direction='above'):
    if direction=='above': return (data>threshold).sum('time')
    if direction=='below': return (data<threshold).sum('time')
    raise ValueError("direction must be 'above' or 'below'")

## 10. Threshold frequency change

In [ ]:
def threshold_frequency_change(data,threshold,direction='above'):
    b=exceedance_frequency(select_period(data,BASELINE_START,BASELINE_END),threshold,direction)
    f=exceedance_frequency(select_period(data,FUTURE_START,FUTURE_END),threshold,direction)
    return f-b

## 11. Relative threshold-frequency change

In [ ]:
def threshold_relative_change(data,threshold,direction='above'):
    b=exceedance_frequency(select_period(data,BASELINE_START,BASELINE_END),threshold,direction)
    f=exceedance_frequency(select_period(data,FUTURE_START,FUTURE_END),threshold,direction)
    return 100*(f-b)/b.where(b>0)

## 12. Heat threshold example

In [ ]:
HEAT_THRESHOLD_C=35.0
# heat_change=threshold_frequency_change(tasmax_data,HEAT_THRESHOLD_C,'above')

## 13. Drought threshold example

In [ ]:
DROUGHT_THRESHOLD=-1.5
# drought_change=threshold_frequency_change(spei3_data,DROUGHT_THRESHOLD,'below')

## 14. Precipitation threshold example

In [ ]:
HEAVY_RAIN_THRESHOLD=50.0
# heavy_rain_change=threshold_frequency_change(pr_data,HEAVY_RAIN_THRESHOLD,'above')

## 15. Percentile intensity

In [ ]:
def percentile_intensity(data,percentile=.95): return data.quantile(percentile,dim='time')

## 16. Percentile change

In [ ]:
def percentile_change(data,percentile=.95):
    b=select_period(data,BASELINE_START,BASELINE_END).quantile(percentile,dim='time')
    f=select_period(data,FUTURE_START,FUTURE_END).quantile(percentile,dim='time')
    return f-b

## 17. Annual statistics

In [ ]:
def annual_mean(data): return data.groupby('time.year').mean('time')
def annual_sum(data): return data.groupby('time.year').sum('time')
def annual_max(data): return data.groupby('time.year').max('time')
def annual_min(data): return data.groupby('time.year').min('time')

## 18. Seasonal statistics

In [ ]:
def seasonal_mean(data): return data.groupby('time.season').mean('time')
def seasonal_sum(data): return data.groupby('time.season').sum('time')
def seasonal_max(data): return data.groupby('time.season').max('time')

## 19. Area-weighted mean

In [ ]:
def area_weighted_mean(data):
    lat='lat' if 'lat' in data.coords else 'latitude'
    lon='lon' if 'lon' in data.dims else 'longitude'
    w=np.cos(np.deg2rad(data[lat]))
    return data.weighted(w).mean(dim=[lat,lon])

## 20. Regional baseline/future summary

In [ ]:
def regional_summary(data):
    b=area_weighted_mean(select_period(data,BASELINE_START,BASELINE_END)).mean('time')
    f=area_weighted_mean(select_period(data,FUTURE_START,FUTURE_END)).mean('time')
    return pd.Series({'baseline':float(b),'future':float(f),'absolute_change':float(f-b)})

## 21. Linear trend

In [ ]:
def linear_trend(values,years):
    values=np.asarray(values,float); years=np.asarray(years,float); ok=np.isfinite(values)&np.isfinite(years)
    return np.nan if ok.sum()<3 else np.polyfit(years[ok],values[ok],1)[0]

## 22. Regional annual trend

In [ ]:
def regional_annual_trend(data):
    a=area_weighted_mean(data).groupby('time.year').mean('time')
    return linear_trend(a.values,a.year.values)

## 23. Model trends

In [ ]:
def model_annual_trends(model_data):
    a=model_data.groupby('time.year').mean('time')
    return xr.apply_ufunc(linear_trend,a,a['year'],input_core_dims=[['year'],['year']],output_core_dims=[[]],vectorize=True,dask='parallelized',output_dtypes=[float])

## 24. Trend agreement

In [ ]:
def trend_agreement(trends): return xr.Dataset({'positive_fraction':(trends>0).mean('model'),'negative_fraction':(trends<0).mean('model')})

## 25. Longest hot spell

In [ ]:
def longest_run_1d(values,threshold):
    mask=np.asarray(values)>threshold; best=current=0
    for x in mask:
        current=current+1 if x else 0; best=max(best,current)
    return best

## 26. Spatial longest hot spell

In [ ]:
def longest_hot_spell(tasmax,threshold=35):
    return xr.apply_ufunc(longest_run_1d,tasmax,kwargs={'threshold':threshold},input_core_dims=[['time']],output_core_dims=[[]],vectorize=True,dask='parallelized',output_dtypes=[int])

## 27. Wet-day mask

In [ ]:
WET_DAY_THRESHOLD=1.0
def wet_day_mask(pr,threshold=WET_DAY_THRESHOLD): return pr>=threshold

## 28. Frequency ratio

In [ ]:
def frequency_ratio(baseline_frequency,future_frequency): return future_frequency/baseline_frequency.where(baseline_frequency>0)

## 29. Frequency change in percentage points

In [ ]:
def frequency_change_percentage_points(baseline_frequency,future_frequency): return 100*(future_frequency-baseline_frequency)

## 30. Robust change mask

In [ ]:
def robust_change_mask(change_by_model,minimum_agreement=.80):
    agreement=xr.concat([(change_by_model>0).mean('model'),(change_by_model<0).mean('model')],dim='direction').max('direction')
    return agreement>=minimum_agreement

## 31. Mask uncertain changes

In [ ]:
def mask_uncertain_changes(change_mean,change_by_model,minimum_agreement=.80): return change_mean.where(robust_change_mask(change_by_model,minimum_agreement))

## 32. Hazard normalization

In [ ]:
def minmax_normalize(data,lower=None,upper=None):
    lower=float(data.min()) if lower is None else lower; upper=float(data.max()) if upper is None else upper
    if upper==lower: return xr.zeros_like(data)
    return ((data-lower)/(upper-lower)).clip(0,1)

## 33. Hazard score

In [ ]:
def hazard_score(*components):
    if not components: raise ValueError('At least one component is required.')
    return sum(components)/len(components)

## 34. Exposure alignment

In [ ]:
def align_spatial(climate,exposure): return xr.align(climate,exposure,join='inner')

## 35. Exposure normalization

In [ ]:
def normalize_exposure(exposure): return minmax_normalize(exposure)

## 36. Vulnerability normalization

In [ ]:
def normalize_vulnerability(vulnerability): return minmax_normalize(vulnerability)

## 37. Simple normalized risk index

Risk is commonly framed as Hazard × Exposure × Vulnerability. The formulation must be justified for the specific study.

In [ ]:
def risk_index(hazard,exposure,vulnerability):
    h,e,v=xr.align(hazard,exposure,vulnerability,join='inner')
    return h*e*v

## 38. Risk change

In [ ]:
def risk_change(baseline_risk,future_risk): return future_risk-baseline_risk

## 39. Relative risk change

In [ ]:
def relative_risk_change(baseline_risk,future_risk): return 100*(future_risk-baseline_risk)/baseline_risk.where(np.abs(baseline_risk)>1e-12)

## 40. Population-weighted hazard

In [ ]:
def population_weighted_mean(hazard,population):
    h,p=xr.align(hazard,population,join='inner')
    return (h*p).sum()/p.sum()

## 41. Ensemble threshold agreement

In [ ]:
def model_increase_fraction(change_by_model): return (change_by_model>0).mean('model')
def model_decrease_fraction(change_by_model): return (change_by_model<0).mean('model')

## 42. Scenario uncertainty

Scenario uncertainty requires multiple forcing scenarios. It should not be inferred from a single scenario.

In [ ]:
def scenario_difference(scenario_a,scenario_b): return scenario_b-scenario_a

## 43. Uncertainty decomposition

Separate conceptually: internal variability, model uncertainty, and scenario uncertainty. The present workflow directly estimates model spread; the other components require suitable data/designs.

## 44. Observed versus simulated anomalies

In [ ]:
def baseline_anomaly(data): return data-data.sel(time=slice(BASELINE_START,BASELINE_END)).mean('time')

## 45. Anomaly difference

In [ ]:
def anomaly_difference(observed,model):
    o,m=xr.align(baseline_anomaly(observed),baseline_anomaly(model),join='inner')
    return m-o

## 46. Distributional validation

In [ ]:
def percentile_summary(data,percentiles=(1,5,10,50,90,95,99)): return data.quantile(np.asarray(percentiles)/100,dim='time')
def percentile_bias(reference,simulation,percentiles=(1,5,10,50,90,95,99)):
    q=np.asarray(percentiles)/100
    return simulation.quantile(q,dim='time')-reference.quantile(q,dim='time')

## 47. Upper-tail frequency

In [ ]:
def upper_tail_frequency(data,threshold): return (data>threshold).mean('time')

## 48. Climate-change signal preservation

In [ ]:
def period_change(data): return period_mean(data,FUTURE_START,FUTURE_END)-period_mean(data,BASELINE_START,BASELINE_END)

## 49. Mean versus median ensemble

In [ ]:
def compare_mean_median(data): return data.mean('model'),data.median('model')

## 50. P10–P90 model spread

In [ ]:
def p10_p90_spread(data):
    q=data.quantile([.10,.90],dim='model')
    return q.sel(quantile=.90)-q.sel(quantile=.10)

## 51. Ensemble time-series plot

In [ ]:
def plot_ensemble_timeseries(data,title='Multi-model ensemble',ylabel='Value'):
    y=area_weighted_mean(data.mean('model'))
    plt.figure(figsize=(12,5)); plt.plot(y.time,y); plt.title(title); plt.xlabel('Time'); plt.ylabel(ylabel); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 52. Spatial change map

In [ ]:
def plot_change_map(change,title='Climate change'):
    plt.figure(figsize=(10,5)); change.plot(); plt.title(title); plt.tight_layout(); plt.show()

## 53. Agreement map

In [ ]:
def plot_agreement_map(agreement,title='Model agreement'):
    plt.figure(figsize=(10,5)); agreement.plot(vmin=0,vmax=1); plt.title(title); plt.tight_layout(); plt.show()

## 54. Uncertainty map

In [ ]:
def plot_uncertainty_map(spread,title='Model spread'):
    plt.figure(figsize=(10,5)); spread.plot(); plt.title(title); plt.tight_layout(); plt.show()

## 55. Historical/future distributions

In [ ]:
def plot_distributions(data,bins=30,title='Historical vs future'):
    b=select_period(data,BASELINE_START,BASELINE_END).values.ravel(); f=select_period(data,FUTURE_START,FUTURE_END).values.ravel()
    b=b[np.isfinite(b)]; f=f[np.isfinite(f)]
    plt.figure(figsize=(9,5)); plt.hist(b,bins=bins,alpha=.5,label='Baseline'); plt.hist(f,bins=bins,alpha=.5,label='Future'); plt.legend(); plt.title(title); plt.tight_layout(); plt.show()

## 56. Risk interpretation

Hazard increase does not automatically equal impact increase. Exposure and vulnerability determine how climate hazards translate into potential impacts.

## 57. Exposure datasets

Possible exposure variables include population, cropland, built-up area, infrastructure, and economic assets. Spatial resolution and reference year must be documented.

## 58. Vulnerability datasets

Vulnerability indices may use socioeconomic, demographic, infrastructure, or adaptive-capacity variables. Variables should be theoretically justified before aggregation.

## 59. Attribution caution

A comparison between observations and model projections is not by itself a formal event-attribution probability statement. Formal attribution generally requires an appropriate counterfactual/factual framework or equivalent probabilistic design.

## 60. Sensitivity analysis

Repeat key results using alternative bias-correction methods, ensemble weighting, baseline periods, thresholds, and model subsets.

In [ ]:
SENSITIVITY_CASES=['bias_correction_method','baseline_period','ensemble_weighting','threshold','model_subset','exposure_dataset','vulnerability_definition']
for x in SENSITIVITY_CASES: print('-',x)

## 61. Summary table

In [ ]:
def indicator_summary(data,indicator):
    b=float(area_weighted_mean(select_period(data,BASELINE_START,BASELINE_END)).mean('time')); f=float(area_weighted_mean(select_period(data,FUTURE_START,FUTURE_END)).mean('time'))
    return {'indicator':indicator,'baseline':b,'future':f,'absolute_change':f-b,'relative_change_percent':100*(f-b)/b if abs(b)>1e-12 else np.nan}

## 62. Save table

In [ ]:
def save_table(df,filename):
    path=TABLE_DIR/filename; df.to_csv(path,index=False); print('Saved:',path)

## 63. Save NetCDF

In [ ]:
def save_result(data,filename):
    path=OUTPUT_DIR/filename; data.to_netcdf(path); print('Saved:',path)

## 64. Add metadata

In [ ]:
def add_metadata(data,indicator,scenario):
    data=data.copy(); data.attrs.update({'indicator':indicator,'scenario':scenario,'baseline_period':f'{BASELINE_START} to {BASELINE_END}','future_period':f'{FUTURE_START} to {FUTURE_END}','analysis_type':'Climate impact attribution and risk analysis'}); return data

## 65. Main analysis versus sensitivity

Main result: corrected model members → equal-weight ensemble → change → spread → agreement. Sensitivity: weighted ensembles and alternative correction/threshold choices.

## 66. Uncertainty reporting

Report central tendency, model spread, and agreement together. Model P10–P90 should be described as the distribution of model outcomes unless a formal probabilistic interpretation is justified.

## 67. Publication figure checklist

Include units, baseline/future periods, scenario, indicator name, spatial domain, model count, and an explanation of any masking or agreement threshold.

## 68. Reproducibility checklist

Record input files, model members, correction method, calibration period, thresholds, exposure/vulnerability datasets, software environment, and output filenames.

## 69. Final production sequence

Preprocessing → indicators → bias correction → ensemble → hazard change → thresholds → uncertainty/agreement → exposure → vulnerability → risk → sensitivity analysis → publication outputs.

## 70. Final research checklist

- [ ] Periods documented
- [ ] Units verified
- [ ] Model members documented
- [ ] Independent validation completed
- [ ] Model spread retained
- [ ] Thresholds justified
- [ ] Area weighting considered
- [ ] Exposure aligned
- [ ] Vulnerability justified
- [ ] Sensitivity analysis completed
- [ ] NetCDF/CSV outputs saved
- [ ] Attribution claims match the methodology

# Conclusion

Notebook 07 provides the bridge from climate indicators and ensemble projections to climate-impact and risk analysis. The recommended approach is to quantify hazard changes first, preserve model uncertainty and agreement, and only then combine climate information with exposure and vulnerability data.